# Contract Risk Scoring Engine — CUAD Fine-Tuning
### Fresnel Fabian | MPS Applied Machine Intelligence | Northeastern Roux Institute

---

This notebook trains and evaluates the full contract risk scoring pipeline using:
- **Your files:** `train.py`, `utils.py`, `evaluate.py`, `category_descriptions.csv`
- **Your data:** `data.zip` → `train_separate_questions.json`, `test.json`
- **Model:** RoBERTa-base fine-tuned on CUAD (extractive QA — span extraction)

**What the model does:** Given a contract and a clause question like *"What are the liability limitations?"*, it extracts the exact text span from the contract. If the clause doesn't exist, it returns empty. This is why it can resolve Cap vs Uncapped Liability — it reads the full clause span in context.

**Data (from your data.zip):**
| Split | Contracts | QA Pairs | Positive | Negative |
|---|---|---|---|---|
| Train | 408 | 22,450 | 11,180 (49.8%) | 11,270 (50.2%) |
| Test | 102 | 4,182 | 1,244 (29.7%) | 2,938 (70.3%) |

**Before running:** `Runtime → Change runtime type → T4 GPU`  
**Expected time:** ~90 min on T4 · ~25 min on A100

## Step 1 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:       ', torch.cuda.get_device_name(0))
    print('VRAM:         ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('Go to Runtime → Change runtime type → GPU and re-run')

GPU available: True
Device:        NVIDIA A100-SXM4-40GB
VRAM:          42.4 GB


## Step 2 — Install Dependencies

In [ ]:
# Pin transformers to 4.12 — matches train.py's imports
# (AutoModelForQuestionAnswering, squad_convert_examples_to_features,
#  SquadV1Processor, SquadV2Processor all available at this version)
!pip install -q 'transformers==4.12.0' 'tokenizers==0.10.3'
!pip install -q tensorboard tensorboardX scikit-learn pandas

import transformers, torch
print('transformers:', transformers.__version__)
print('torch:       ', torch.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 12.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 64.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.3 MB/s eta 0:00:00
transformers: 5.0.0
torch:        2.10.0+cu128


## Step 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/contract_risk_cuad'
for d in [BASE, f'{BASE}/data', f'{BASE}/trained_models/roberta-base',
          f'{BASE}/results', f'{BASE}/cache']:
    os.makedirs(d, exist_ok=True)
print('Drive mounted. Project folder:', BASE)

Mounted at /content/drive
Drive mounted. Project folder: /content/drive/MyDrive/contract_risk_cuad


## Step 4 — Upload Your Project Files

Upload these 5 files (select all at once when the dialog opens):
- `train.py`
- `utils.py`
- `evaluate.py`
- `category_descriptions.csv`
- `data.zip`

If you already uploaded them to Drive in a previous session, set `ALREADY_UPLOADED = True`.

In [ ]:
ALREADY_UPLOADED = False  # Set True if files are already in BASE folder on Drive

import shutil, os

if not ALREADY_UPLOADED:
    from google.colab import files
    print('Select train.py, utils.py, evaluate.py, category_descriptions.csv, data.zip')
    uploaded = files.upload()   # file picker dialog
    for fname in uploaded:
        shutil.copy(fname, f'{BASE}/{fname}')
        print(f'  Saved: {fname}')

# Copy scripts to /content so Python can import them
for fname in ['train.py', 'utils.py', 'evaluate.py', 'category_descriptions.csv']:
    src = f'{BASE}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{fname}')
    else:
        print(f'MISSING: {src} — did you upload it?')

os.chdir('/content')
print('\nFiles in /content/:', [f for f in os.listdir() if f.endswith(('.py','.csv'))])

Select train.py, utils.py, evaluate.py, category_descriptions.csv, data.zip


Saving category_descriptions.csv to category_descriptions.csv
Saving data.zip to data (1).zip
Saving evaluate.py to evaluate.py
Saving train.py to train.py
Saving utils.py to utils.py
  Saved: category_descriptions.csv
  Saved: data (1).zip
  Saved: evaluate.py
  Saved: train.py
  Saved: utils.py

Files in /content/: ['utils.py', 'evaluate.py', 'category_descriptions.csv', 'train.py']


## Step 5 — Extract Data

In [ ]:
import zipfile, os, json

DATA_DIR   = f'{BASE}/data'
TRAIN_FILE = f'{DATA_DIR}/train_separate_questions.json'
TEST_FILE  = f'{DATA_DIR}/test.json'

if not os.path.exists(TRAIN_FILE):
    zip_path = f'{BASE}/data.zip'
    print(f'Extracting {zip_path} ...')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
    print('Done.')
else:
    print('Data already extracted.')

# Print stats to confirm
for path, label in [(TRAIN_FILE, 'Train'), (TEST_FILE, 'Test')]:
    with open(path) as f:
        d = json.load(f)
    contracts = d['data']
    qa_pairs  = [q for c in contracts for p in c['paragraphs'] for q in p['qas']]
    pos       = sum(1 for q in qa_pairs if q['answers'])
    print(f'{label}: {len(contracts)} contracts, {len(qa_pairs)} QA pairs '
          f'({pos} positive / {len(qa_pairs)-pos} negative)')

Extracting /content/drive/MyDrive/contract_risk_cuad/data.zip ...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/contract_risk_cuad/data.zip'

## Step 6 — Fix One Colab Compatibility Issue in train.py

`train.py` calls `scheduler.get_lr()` which was deprecated in PyTorch 1.9+. One line patch.

In [ ]:
with open('/content/train.py') as f:
    src = f.read()

if 'scheduler.get_lr()[0]' in src:
    src = src.replace('scheduler.get_lr()[0]', 'scheduler.get_last_lr()[0]')
    with open('/content/train.py', 'w') as f:
        f.write(src)
    print('Patched: scheduler.get_lr()[0] → scheduler.get_last_lr()[0]')
else:
    print('No patch needed — already compatible.')

# Quick import check
import sys; sys.path.insert(0, '/content')
import utils, evaluate
print('utils.py import:    OK')
print('evaluate.py import: OK')

## Step 7 — Train

Runs your `train.py` with the same parameters as `run.sh`, adapted for a single T4 GPU.

**What `train.py` does internally:**
1. Tokenizes all 22,450 QA pairs (question + full contract context, max 512 tokens)
2. Applies `get_balanced_dataset()` — samples negatives to match positive count
3. Fine-tunes `AutoModelForQuestionAnswering` to predict start/end positions of clause spans
4. Runs evaluation on test.json after training
5. Writes `nbest_predictions_.json` to output dir — used by `evaluate.py`

In [ ]:
import os, torch

OUTPUT_DIR = f'{BASE}/trained_models/roberta-base'
CACHE_DIR  = f'{BASE}/cache'

# T4 has 16 GB VRAM. batch_size=16 fits safely with 512 token sequences.
# run.sh uses batch_size=40 — that's for a multi-GPU A100 setup.
# We use batch_size=8, accum_steps=5 → same effective batch of 40.
BATCH_SIZE  = 8
ACCUM_STEPS = 5    # effective batch = 8 * 5 = 40 (matches run.sh)

cmd = f"""python /content/train.py \\
    --output_dir {OUTPUT_DIR} \\
    --model_type roberta \\
    --model_name_or_path roberta-base \\
    --train_file {TRAIN_FILE} \\
    --predict_file {TEST_FILE} \\
    --do_train \\
    --do_eval \\
    --version_2_with_negative \\
    --learning_rate 1e-4 \\
    --num_train_epochs 4 \\
    --per_gpu_train_batch_size={BATCH_SIZE} \\
    --per_gpu_eval_batch_size=16 \\
    --gradient_accumulation_steps={ACCUM_STEPS} \\
    --max_seq_length 512 \\
    --max_answer_length 512 \\
    --doc_stride 256 \\
    --max_query_length 64 \\
    --n_best_size 20 \\
    --save_steps 1000 \\
    --cache_dir {CACHE_DIR} \\
    --overwrite_output_dir \\
    --seed 42"""

print('Training command:')
print(cmd)
print()
print('Starting training — T4 estimate ~90 min for 4 epochs')
print('=' * 60)

os.system(cmd)

## Step 8 — Evaluate with evaluate.py

In [ ]:
# Run evaluate.py exactly as its __main__ block does
import sys, json, os
sys.path.insert(0, '/content')
import evaluate as ev

MODEL_PATH = f'{BASE}/trained_models/roberta-base'
SAVE_DIR   = f'{BASE}/results'
os.makedirs(SAVE_DIR, exist_ok=True)

# Load ground truth from test.json
gt_dict = ev.get_answers(ev.load_json(TEST_FILE))
print(f'Ground truth entries: {len(gt_dict)}')

# Run evaluation (uses nbest_predictions_.json written by train.py)
print('\nRunning evaluate.py...')
results = ev.get_results(MODEL_PATH, gt_dict, verbose=True)
print()
print('Results:', results)

# Save
save_path = f'{SAVE_DIR}/roberta-base.json'
with open(save_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved to: {save_path}')

## Step 9 — Per-Category Results

In [ ]:
import sys, json
import numpy as np
sys.path.insert(0, '/content')
import evaluate as ev

MODEL_PATH = f'{BASE}/trained_models/roberta-base'
gt_dict    = ev.get_answers(ev.load_json(TEST_FILE))
pred_dict  = ev.load_json(f'{MODEL_PATH}/nbest_predictions_.json')

# Load the 41 category names from category_descriptions.csv
qtype_dict = ev.get_questions_from_csv()
categories = sorted(qtype_dict.keys())

print(f'Evaluating {len(categories)} clause categories...')
print()

cat_results = []
for cat in categories:
    try:
        precs, recs, confs = ev.get_precisions_recalls(pred_dict, gt_dict, category=cat)
        p80, _ = ev.get_prec_at_recall(precs, recs, confs, recall_thresh=0.80)
        p90, _ = ev.get_prec_at_recall(precs, recs, confs, recall_thresh=0.90)
        aupr   = ev.get_aupr(precs, recs)
        # Count test positives for this category
        n_pos  = sum(1 for k, v in gt_dict.items() if cat in k and len(v) > 0)
        cat_results.append({'category': cat, 'aupr': aupr,
                             'prec_at_80': p80, 'prec_at_90': p90, 'n_positive': n_pos})
    except Exception as e:
        print(f'  Skipped {cat}: {e}')

# Print table sorted by AUPR
cat_results.sort(key=lambda x: x['aupr'], reverse=True)
print(f'{"Category":<40} {"AUPR":>6}  {"P@80%":>6}  {"P@90%":>6}  {"N+":>4}')
print('-' * 68)
for r in cat_results:
    bar  = '█' * int(r['aupr'] * 20)
    flag = '  ⚠' if r['aupr'] < 0.40 else ''
    print(f"{r['category']:<40} {r['aupr']:>6.3f}  {r['prec_at_80']:>6.3f}  "
          f"{r['prec_at_90']:>6.3f}  {r['n_positive']:>4}  {bar}{flag}")

auprs = [r['aupr'] for r in cat_results]
print(f'\nMean AUPR:   {np.mean(auprs):.4f}')
print(f'Median AUPR: {np.median(auprs):.4f}')
print(f'Best:  {cat_results[0]["category"]} ({cat_results[0]["aupr"]:.3f})')
print(f'Worst: {cat_results[-1]["category"]} ({cat_results[-1]["aupr"]:.3f})')

# Save per-category results
with open(f'{BASE}/results/per_category.json', 'w') as f:
    json.dump(cat_results, f, indent=2)
print(f'Saved: {BASE}/results/per_category.json')

## Step 10 — Precision-Recall Curve + Model Comparison Chart

In [ ]:
import sys, json
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '/content')
import evaluate as ev

NAVY, TEAL, AMBER, RED, BLUE = '#0F2444','#0D9488','#B45309','#B91C1C','#1D6FA4'

MODEL_PATH = f'{BASE}/trained_models/roberta-base'
gt_dict    = ev.get_answers(ev.load_json(TEST_FILE))
pred_dict  = ev.load_json(f'{MODEL_PATH}/nbest_predictions_.json')

precs, recs, confs = ev.get_precisions_recalls(pred_dict, gt_dict)
proc_precs = ev.process_precisions(precs)
aupr = ev.get_aupr(precs, recs)
p80, c80 = ev.get_prec_at_recall(precs, recs, confs, 0.80)
p90, c90 = ev.get_prec_at_recall(precs, recs, confs, 0.90)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Precision-Recall curve
ax = axes[0]
ax.plot(recs, proc_precs, color=NAVY, lw=2.5,
        label=f'RoBERTa-base on CUAD  (AUPR = {aupr:.3f})')
ax.fill_between(recs, proc_precs, alpha=0.07, color=NAVY)
ax.axvline(0.80, color=AMBER, ls='--', lw=1.5, label=f'P@80% recall = {p80:.3f}')
ax.axvline(0.90, color=RED,   ls='--', lw=1.5, label=f'P@90% recall = {p90:.3f}')
ax.scatter([0.80, 0.90], [p80, p90], s=80, zorder=5, color=[AMBER, RED])
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
ax.set_xlabel('Recall', fontsize=11); ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curve\n'
             'Jaccard ≥ 0.5 match | CUAD test set (102 contracts)', fontsize=10)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Right: Per-category AUPR
ax = axes[1]
with open(f'{BASE}/results/per_category.json') as f:
    cat_results = json.load(f)
cat_results.sort(key=lambda x: x['aupr'])
names   = [r['category'][:25] for r in cat_results]
auprs_c = [r['aupr'] for r in cat_results]
bar_colors = [TEAL if a >= 0.7 else AMBER if a >= 0.4 else RED for a in auprs_c]

ax.barh(range(len(names)), auprs_c, color=bar_colors, alpha=0.85)
ax.axvline(np.mean(auprs_c), color=NAVY, ls='--', lw=1.5,
           label=f'Mean = {np.mean(auprs_c):.3f}')
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
ax.set_xlabel('AUPR', fontsize=11); ax.set_xlim(0, 1)
ax.set_title('AUPR per Clause Category\n'
             '(teal ≥ 0.7 · amber ≥ 0.4 · red < 0.4)', fontsize=10)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Contract Risk Scoring Engine — RoBERTa-base on CUAD',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()

fig_path = f'{BASE}/results/evaluation_charts.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## Step 11 — Run Inference on a Contract

In [ ]:
import sys, torch
sys.path.insert(0, '/content')
import evaluate as ev
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

MODEL_PATH = f'{BASE}/trained_models/roberta-base'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Loading model from {MODEL_PATH} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
model     = AutoModelForQuestionAnswering.from_pretrained(MODEL_PATH).to(DEVICE)
model.eval()
print('Model loaded.')

# Load the 41 clause questions from category_descriptions.csv
qtype_dict = ev.get_questions_from_csv()

CLAUSE_RISK_WEIGHT = {
    'Uncapped Liability': 10, 'Limitation Of Liability': 9, 'Cap On Liability': 9,
    'Ip Ownership Assignment': 9, 'Joint Ip Ownership': 8, 'Change Of Control': 8,
    'Anti-Assignment': 7, 'Liquidated Damages': 7, 'Source Code Escrow': 7,
    'Unlimited License': 6, 'Non-Compete': 6, 'Audit Rights': 6,
    'Unilateral Termination': 6, 'Covenant Not To Sue': 6, 'Most Favored Nation': 6,
}

@torch.no_grad()
def score_contract(contract_text, conf_threshold=0.05):
    """
    Runs all 41 clause questions through the fine-tuned RoBERTa model.
    Returns detected clauses with extracted text spans and confidence.
    """
    detected = []

    for clause_type, question in qtype_dict.items():
        # Tokenize question + contract (sliding window for long contracts)
        enc = tokenizer(
            question, contract_text,
            max_length=512, truncation='only_second', stride=256,
            return_overflowing_tokens=True, return_offsets_mapping=True,
            padding='max_length', return_tensors='pt'
        )
        offsets   = enc.pop('offset_mapping')
        enc.pop('overflow_to_sample_mapping', None)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        out = model(**enc)
        s_logits, e_logits = out.start_logits, out.end_logits

        import torch.nn.functional as F
        best_conf, best_text = 0.0, ''

        for i in range(len(s_logits)):
            s_idx = s_logits[i].argmax().item()
            e_idx = e_logits[i].argmax().item()
            if s_idx > e_idx: continue

            conf = (F.softmax(s_logits[i], dim=-1)[s_idx] *
                    F.softmax(e_logits[i], dim=-1)[e_idx]).item()

            if conf > best_conf:
                off = offsets[i]
                cs, ce = off[s_idx][0].item(), off[e_idx][1].item()
                if cs == ce == 0: continue   # special token
                best_conf = conf
                best_text = contract_text[cs:ce].strip()

        if best_conf >= conf_threshold and best_text:
            detected.append({
                'clause':      clause_type,
                'text':        best_text,
                'confidence':  round(best_conf, 4),
                'risk_weight': CLAUSE_RISK_WEIGHT.get(clause_type, 3),
            })

    detected.sort(key=lambda x: x['risk_weight'] * x['confidence'], reverse=True)

    # Risk score
    import numpy as np
    if detected:
        w = np.array([d['risk_weight'] for d in detected], dtype=float)
        c = np.array([d['confidence']  for d in detected], dtype=float)
        raw   = float(np.dot(w / w.sum(), c))
        boost = sum(d['confidence'] * d['risk_weight'] for d in detected if d['risk_weight'] >= 7)
        score = min(100, int(raw * 80 + boost * 3))
    else:
        score = 0

    if score < 30:   triage = 'AUTO-CLEAR'
    elif score < 65: triage = 'FLAG FOR REVIEW'
    else:            triage = 'URGENT REVIEW'

    return {'score': score, 'triage': triage, 'clauses': detected}


# ── Sample: High-risk contract ─────────────────────────────────────────────────
HIGH_RISK = """SOFTWARE AS A SERVICE AGREEMENT

This Agreement is entered into between TechVendor Inc. ("Vendor") and Customer Corp.
All intellectual property created under this Agreement shall be assigned to Vendor.
There shall be no cap on liability under any circumstances — Vendor's liability
shall not be limited or capped for any damages. The agreement auto-renews annually
unless terminated with 90 days prior written notice. Governing law is Delaware.
Customer agrees to a non-compete for 24 months post-termination. Liquidated damages
of $500,000 apply for breach. Any change of control of Customer requires Vendor consent.
"""

LOW_RISK = """NON-DISCLOSURE AGREEMENT

This Non-Disclosure Agreement is entered into as of January 1, 2026 between Party A
and Party B. Both parties agree to keep all disclosed information strictly confidential.
Governed by Delaware law. Term is two years. Either party may terminate with 30 days notice.
"""

for label, contract in [('HIGH-RISK SaaS Agreement', HIGH_RISK),
                         ('LOW-RISK NDA',            LOW_RISK)]:
    result = score_contract(contract)
    print(f'{'='*60}')
    print(f'{label}')
    print(f'Risk Score: {result["score"]}/100   Triage: {result["triage"]}')
    print(f'Clauses detected: {len(result["clauses"])}')
    if result['clauses']:
        print('Top flags:')
        for d in result['clauses'][:5]:
            flag = '  ⚠' if d['risk_weight'] >= 8 else ''
            print(f"  [{d['risk_weight']:>2}/10] {d['clause']:<35} "
                  f"{d['confidence']:.3f}  {repr(d['text'][:55])}{flag}")
    print()

## Step 12 — Final Summary

In [ ]:
import json, os
import numpy as np

with open(f'{BASE}/results/roberta-base.json') as f:
    overall = json.load(f)
with open(f'{BASE}/results/per_category.json') as f:
    per_cat = json.load(f)

auprs = [r['aupr'] for r in per_cat]

print('=' * 65)
print('CONTRACT RISK SCORING ENGINE — FINAL RESULTS')
print('Model: RoBERTa-base fine-tuned on CUAD')
print('=' * 65)
print(f'  AUPR (overall):          {overall["aupr"]:.4f}')
print(f'  Precision @ 80% Recall:  {overall["prec_at_80_recall"]:.4f}')
print(f'  Precision @ 90% Recall:  {overall["prec_at_90_recall"]:.4f}')
print(f'  Mean category AUPR:      {np.mean(auprs):.4f}')
print(f'  Categories with AUPR≥0.7: {sum(1 for a in auprs if a >= 0.7)}/41')
print(f'  Categories with AUPR<0.4: {sum(1 for a in auprs if a < 0.4)}/41')
print()
print('  Top 5 best-performing clause types:')
for r in sorted(per_cat, key=lambda x: -x['aupr'])[:5]:
    print(f"    {r['category']:<38} AUPR={r['aupr']:.3f}  (n+={r['n_positive']})")
print()
print('  5 hardest clause types:')
for r in sorted(per_cat, key=lambda x: x['aupr'])[:5]:
    print(f"    {r['category']:<38} AUPR={r['aupr']:.3f}  (n+={r['n_positive']})")
print()
print('  Saved to Google Drive:')
for fname in os.listdir(f'{BASE}/results'):
    path = f'{BASE}/results/{fname}'
    print(f'    {fname}  ({os.path.getsize(path)//1024} KB)')